Characterize the 2×2×1 training CSS before model evaluation, where $w_i$ denotes the multiplicity of symmetry-equivalent realizations and $N_U$ and $N_{Np}$ count vacancies on the metal and chalcogen sublattices rather than chemical elements.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from collections import OrderedDict

In [ ]:
data = pd.read_pickle("data/train_ccs-gnn_predictions_dft_subset.pkl.gz")
masks = ['Me3X6_exactly']
space_groups = dict(zip(data['Space_group_no'], data['Space_group_symbol']))
space_groups = OrderedDict(sorted(space_groups.items(), reverse=True))

Plot space-group distribution of training CCS structures with $Me_3X_6$ composition.

In [ ]:
plt.figure(figsize=(10, 2))
plt.hist([data[data[mask]]['Space_group_no'] for mask in masks], label=masks, bins=np.linspace(0, 190, 50), edgecolor='k')
plt.yscale('log')
plt.xlabel('Space group')
plt.ylabel('Number of structures')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

For training CCS entries, plot pairwise relationships between compositions, space groups, weights, and the number of defect positions in a 2×2×1 supercell. W_nsites and Se_nsites denote the number of positions occupied by W and Se, respectively. U_nsites (vacancies on the metal sublattice) and Np_nsites (vacancies on the chalcogen sublattice) are provided.

In [ ]:
sns.pairplot(data[['W_nsites', 'U_nsites', 'Se_nsites', 'Np_nsites', 'Space_group_no', 'Space_group_symbol', 'Weight', 'defect_nsites']], 
             hue='Space_group_symbol', hue_order=list(space_groups.values()), palette='crest', corner=True)
plt.tight_layout()
plt.show()

Plot space-group distribution in the training set versus number of vacancies and substitutions in the $WSe_2$ monolayer; box numbers are counts of inequivalent structures.

In [ ]:
plt.figure(figsize=(9, 7))
numbers = data.groupby(['defect_nsites', 'Space_group_no']).size().to_frame()\
          .rename(mapper={0: 'Number_of_structures'}, axis=1).reset_index()\
          .pivot(columns='defect_nsites', index='Space_group_no', values='Number_of_structures')
numbers.sort_index(ascending=False, inplace=True)
numbers = numbers.T
ax = sns.heatmap(np.log10(numbers), cmap='crest', annot=numbers, linewidth=.1, 
                 cbar_kws={'label': 'lg (number of inequivalent structures)', 'drawedges': False, "shrink": 0.6}, fmt='g',
                 xticklabels=list(map(lambda x: f'{space_groups[x]}\n({x})', numbers.columns)))
ax.set(xlabel="Space group", ylabel="Number of defect positions")
ax.xaxis.tick_top()
plt.show()